In [2]:
pip install sentence-transformers

In [3]:
pip install pymorphy3 pymorphy3-dicts-ru

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 21.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [pymorphy3]


In [4]:

import math
import re
from collections import Counter

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import GroupKFold
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import CrossEncoder
import unicodedata
import html
import os, pickle
import torch

In [5]:
STRESS_RE = re.compile("[\u0300\u0301\u0340\u0341]")
TOKEN_RE  = re.compile(r"[а-яёa-z0-9]+")
WEB_GARBAGE = re.compile(
    r"^(главная\s*[›»>].*|смотрите также.*|политика конфиденциальности.*|"
    r"обратная связь.*|читайте также.*|источник.*|реклама.*)$",
    flags=re.IGNORECASE,
)

def clean_text(s):
    if not isinstance(s, str):
        return ""
    s = STRESS_RE.sub("", s).replace("ё", "е").replace("Ё", "Е")
    lines = [ln.strip() for ln in s.splitlines()]
    lines = [ln for ln in lines if ln and not WEB_GARBAGE.match(ln)]
    return re.sub(r"\s+", " ", " ".join(lines)).strip()

# ---------- tokenize ----------
try:
    import pymorphy3 as pymorphy2         # ← единственное изменение
    _morph = pymorphy2.MorphAnalyzer()
    _lemma_cache = {}
    def _lemma(t):
        if t not in _lemma_cache:
            _lemma_cache[t] = _morph.parse(t)[0].normal_form
        return _lemma_cache[t]
    def tokenize(s):
        return [_lemma(t) for t in TOKEN_RE.findall(s.lower())]
    print("pymorphy3: on")
except Exception as e:
    def tokenize(s):
        return TOKEN_RE.findall(s.lower())
    print(f"pymorphy3: off ({e})")

def char3(s):
    toks = TOKEN_RE.findall(s.lower())
    j = " ".join(toks)
    return {j[i:i+3] for i in range(len(j)-2)} if len(j) >= 3 else set()

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def safe_log1p(x):
    try:
        xf = float(x)
    except (TypeError, ValueError):
        return np.nan
    if not np.isfinite(xf) or xf <= -1.0:
        return np.nan
    return math.log1p(xf)

_K1, _B = 1.5, 0.75
def bm25(q_terms, tf, dl, idf, avgdl):
    if dl == 0:
        return 0.0
    s = 0.0
    for t in q_terms:
        f = tf.get(t, 0)
        if f and t in idf:
            s += idf[t] * f * (_K1+1) / (f + _K1*(1 - _B + _B*dl/avgdl))
    return s

def build_idf(token_dict):
    n = len(token_dict)
    df_counts = Counter()
    for toks in token_dict.values():
        df_counts.update(set(toks))
    idf = {t: math.log((n - c + 0.5)/(c + 0.5) + 1.0) for t, c in df_counts.items()}
    avgdl = float(np.mean([len(t) for t in token_dict.values()])) if n else 1.0
    return idf, max(avgdl, 1.0)

def ndcg_at_k(y, s, k=10):
    order = np.argsort(-s, kind="stable")
    gains = y[order][:k].astype(float)
    disc  = 1.0 / np.log2(np.arange(2, len(gains)+2))
    dcg   = float((gains * disc).sum())
    ideal = np.sort(y.astype(float))[::-1][:k]
    d_i   = 1.0 / np.log2(np.arange(2, len(ideal)+2))
    idcg  = float((ideal * d_i).sum())
    return None if idcg == 0 else dcg / idcg

QUESTION_PATTERNS = {
    "q_howmany": r"\bсколько\b",
    "q_when":    r"\b(когда|в каком году|в каком веке)\b",
    "q_where":   r"\b(где|в каком городе|в какой стране|куда)\b",
    "q_who":     r"\b(кто|кому|кем|кого)\b",
    "q_what":    r"\b(что такое|что означает|что значит)\b",
    "q_yesno":   r"\b(был ли|были ли|была ли|было ли|является ли|являлся ли|можно ли|есть ли)\b",
    "q_which":   r"\b(какой|какая|какое|какие|который|которая)\b",
}
QUESTION_PATTERNS = {k: re.compile(p, re.IGNORECASE) for k, p in QUESTION_PATTERNS.items()}

def detect_question_types(q):
    if not isinstance(q, str):
        return {k: 0 for k in QUESTION_PATTERNS}
    return {k: int(bool(p.search(q))) for k, p in QUESTION_PATTERNS.items()}

RE_DIGIT       = re.compile(r"\d")
RE_YEAR        = re.compile(r"\b(1[0-9]{3}|20[0-9]{2})\b")
RE_DIGIT_RANGE = re.compile(r"\d+\s*[–\-—]\s*\d+")
RE_NUMWORD     = re.compile(
    r"\b(один|одна|одно|два|две|три|четыре|пять|шесть|семь|восемь|девять|десять|"
    r"сто|двести|триста|тысяча|миллион|миллиард)\b", re.IGNORECASE)
RE_CAP_WORD    = re.compile(r"\b([А-ЯЁ][а-яё]+)")

def text_signals(s):
    out = dict(has_digit=0, has_year=0, has_digit_range=0, has_numword=0,
               n_cap_words=0, has_cap_inside=0)
    if not isinstance(s, str) or not s:
        return out
    out["has_digit"]       = int(bool(RE_DIGIT.search(s)))
    out["has_year"]        = int(bool(RE_YEAR.search(s)))
    out["has_digit_range"] = int(bool(RE_DIGIT_RANGE.search(s)))
    out["has_numword"]     = int(bool(RE_NUMWORD.search(s)))
    cap = RE_CAP_WORD.findall(s)
    first = s.lstrip().split(maxsplit=1)
    first_word = first[0].rstrip(".,!?;:") if first else ""
    n_cap = len(cap)
    if cap and first_word and cap[0] == first_word:
        n_cap = max(0, n_cap - 1)          # не считаем слово в начале предложения
    out["n_cap_words"]    = n_cap
    out["has_cap_inside"] = int(n_cap > 0)
    return out

pymorphy3: on


In [6]:

EMB_CACHE = "emb_cache.pkl"
CE_CACHE  = "ce_cache.pkl"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"device: {DEVICE}")
EMB_MODEL_NAME = "sergeyzh/rubert-tiny-turbo"
CE_MODEL_NAME  = "ARGA100/ru-reranker-modernbert-small"
EMB_BATCH = 128 if DEVICE == "cuda" else 64
CE_BATCH  = 64  if DEVICE == "cuda" else 32
CE_MAX_LEN = 512
N_FOLDS, K, SEED = 5, 10, 42


OUT_PATH   = "submission.csv"


device: cuda


In [7]:

train = pd.read_csv('train_ran.csv')
test = pd.read_csv('test_ran.csv')
train

,id,query_id,document_id,article_id,query,title,text,passage_index,article_passage_count,article_char_count,ingest_source,relevance
0,r_0000a01ae1fc85409cfc,q_0c47372a2c70b5d45a3e,d_3d8b8c509b5daa75f1a3,a_9cb99738b80787ceb3b0,Сколько частей у фильма Властелин Колец?,Влияние и адаптации произведений Толкина,В 1968 году писатель продал права на экранизац...,4.0,72.0,27472,S0,0
1,r_0000d01ccdb12a83c85e,q_f692ddf7b1cf937cfda9,d_8a4095cfd564a399f67c,a_f0741cd81ca38fbb51c7,Где находится Кременчуг?,Кременчуг,Город является крупным железнодорожным узлом Ц...,38.0,47.0,9783,S3,1
2,r_000385ed0a8e1ebfe0e5,q_06c523053ba77923e44c,d_234baccfeddd33d157ca,a_77984ff4169b47f83448,Были ли краёвцы в России?,Краёвцы,Среди краёвцев выделилась два направления (либ...,17.0,74.0,69424,S0,0
3,r_00054d6cc20063390aca,q_04bd545ca365fa0d00e3,d_91bcf49612c776f41ce5,a_86688ad938f8cc3e4e38,Был ли календарь в Древней Мессопотамии?,Астрономия Древней Греции,Календарь греков был лунно-солнечным. Среди ав...,26.0,131.0,76838,S0,0
4,r_0006e17afc349c6a50bc,q_7be4b60db6538256bd28,d_e0a9f9291be0ede0ab2c,a_e057fa66315d8d70643e,Когда появился английский язык?,Английский язык,Главная › Энциклопедия\n\nСледующий период в р...,15.0,47.0,19511,S0,0
...,...,...,...,...,...,...,...,...,...,...,...,...
31525,r_fff9e56f88ea7680ab5a,q_acf47a23cecc1d052e4c,d_43c66891fa4b281721bd,a_10d8bc50108717f50fc9,Где похоронен Томма́зо Кампане́лла?,"Кампанелла, Томмазо",Перед кончиной Кампанелла в своих посланиях (в...,30.0,92.0,NaN,S1,1
31526,r_fffa2d26888c0653ec84,q_71fb133153ca0daf6b24,d_dd318d59794af86ff169,a_d77f71c9c1612d091c90,Что такое красный террор в России?,Красный террор,"Кра́сный терро́р — комплекс карательных мер, п...",0.0,169.0,65867,S0,1
31527,r_fffc8981bf66a23421c4,q_2a0ae2f39326307b27b3,d_780548683026de212222,a_c970a486aff92a57c862,Когда был выпущен M24?,M24 Чаффи,Несмотря на значительные объёмы поставок новог...,45.0,58.0,35285,S0,0
31528,r_fffc9fb4d797a469764a,q_75ddc3be3fe34231ae04,d_16b764adef5c91c60622,a_fcb189b92ee89f8bca25,Сколько официальных языков в Гонконге?,Языки Гонконга,Официальными письменными языками Гонконга явля...,0.0,6.0,6044,S1,1


In [8]:
def title_na(df):
  df_copy = df.copy()
  df_copy.loc[df_copy['title'].apply(lambda x: len(x) if isinstance(x,str) else 0) < 2,'title'] = ''
  return df_copy
def normalize_passage_index(df):
    df = df.copy()
    src_min = df.groupby('ingest_source')['passage_index'].transform('min')
    df['passage_index_norm'] = df['passage_index'] - src_min
    return df
train = title_na(train)
train = normalize_passage_index(train)
test = title_na(test)
test = normalize_passage_index(test)


In [9]:
STRESS_RE    = re.compile("[\u0300\u0301\u0340\u0341]")
TOKEN_RE     = re.compile(r"[а-яёa-z0-9]+")

INVISIBLE_RE = re.compile("[\u00ad\u180e\u200b-\u200f\u2060-\u2064\u206a-\u206f\ufeff]")
WEIRD_SPACES = re.compile("[\u00a0\u1680\u2000-\u200a\u202f\u205f\u3000]")
DASHES_RE    = re.compile("[\u2010\u2011\u2012\u2013\u2014\u2015\u2212]")
QUOTES_RE    = re.compile("[\u00ab\u00bb\u201c\u201d\u201e\u201f\u2033\u2036]")
APOS_RE      = re.compile("[\u2018\u2019\u201a\u201b\u2032\u2035]")
ELLIPSIS_RE  = re.compile(r"\.{3,}")
CTRL_RE      = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f\x7f]")
TAG_RE       = re.compile(r"</?[a-zA-Z][^<>]{0,500}>")

WEB_GARBAGE = re.compile(
    r"^(главная\s*[›»>].*|смотрите также.*|политика конфиденциальности.*|"
    r"обратная связь.*|читайте также.*|источник.*|реклама.*|"
    r"перейти к (?:содержанию|основному).*|"
    r"подпис(?:ыва|а)йтесь.*|поделиться.*|наверх.*|"
    r"версия для слабовидящих.*|cookie.*|войти.*|регистрация.*)$",
    flags=re.IGNORECASE,
)


def clean_text(s: str, *, map_yo: bool = True) -> str:
    if not isinstance(s, str):
        return ""
    s = unicodedata.normalize("NFC", s)
    s = html.unescape(s)
    s = TAG_RE.sub(" ", s)
    s = STRESS_RE.sub("", s)
    s = INVISIBLE_RE.sub("", s)
    s = CTRL_RE.sub(" ", s)
    if map_yo:
        s = s.replace("ё", "е").replace("Ё", "Е")
    s = WEIRD_SPACES.sub(" ", s)
    s = DASHES_RE.sub("-", s)
    s = QUOTES_RE.sub('"', s)
    s = APOS_RE.sub("'", s)
    s = ELLIPSIS_RE.sub("...", s)
    s = s.replace("\u2028", "\n").replace("\u2029", "\n")
    lines = [ln.strip() for ln in s.splitlines()]
    lines = [ln for ln in lines if ln and not WEB_GARBAGE.match(ln)]
    s = " ".join(lines)
    return re.sub(r"\s+", " ", s).strip()


try:
    import pymorphy3
    _morph = pymorphy3.MorphAnalyzer()
    _lemma_cache = {}
    def _lemma(t):
        if t not in _lemma_cache:
            _lemma_cache[t] = _morph.parse(t)[0].normal_form
        return _lemma_cache[t]
    def tokenize(s):
        return [_lemma(t) for t in TOKEN_RE.findall(s.lower())]
    print("pymorphy3: on")
except Exception as e:
    def tokenize(s):
        return TOKEN_RE.findall(s.lower())
    print(f"pymorphy3: off ({e})")

def char3(s):
    toks = TOKEN_RE.findall(s.lower())
    j = " ".join(toks)
    return {j[i:i+3] for i in range(len(j)-2)} if len(j) >= 3 else set()

def jaccard(a, b):
    if not a or not b:
        return 0.0
    return len(a & b) / len(a | b)

def safe_log1p(x):
    try:
        xf = float(x)
    except (TypeError, ValueError):
        return np.nan
    if not np.isfinite(xf) or xf <= -1.0:
        return np.nan
    return math.log1p(xf)

_K1, _B = 1.5, 0.75
def bm25(q_terms, tf, dl, idf, avgdl):
    if dl == 0:
        return 0.0
    s = 0.0
    for t in q_terms:
        f = tf.get(t, 0)
        if f and t in idf:
            s += idf[t] * f * (_K1+1) / (f + _K1*(1 - _B + _B*dl/avgdl))
    return s

def build_idf(token_dict):
    n = len(token_dict)
    df_counts = Counter()
    for toks in token_dict.values():
        df_counts.update(set(toks))
    idf = {t: math.log((n - c + 0.5)/(c + 0.5) + 1.0) for t, c in df_counts.items()}
    avgdl = float(np.mean([len(t) for t in token_dict.values()])) if n else 1.0
    return idf, max(avgdl, 1.0)

def ndcg_at_k(y, s, k=10):
    order = np.argsort(-s, kind="stable")
    gains = y[order][:k].astype(float)
    disc  = 1.0 / np.log2(np.arange(2, len(gains)+2))
    dcg   = float((gains * disc).sum())
    ideal = np.sort(y.astype(float))[::-1][:k]
    d_i   = 1.0 / np.log2(np.arange(2, len(ideal)+2))
    idcg  = float((ideal * d_i).sum())
    return None if idcg == 0 else dcg / idcg

QUESTION_PATTERNS = {
    "q_howmany": r"\bсколько\b",
    "q_when":    r"\b(когда|в каком году|в каком веке)\b",
    "q_where":   r"\b(где|в каком городе|в какой стране|куда)\b",
    "q_who":     r"\b(кто|кому|кем|кого)\b",
    "q_what":    r"\b(что такое|что означает|что значит)\b",
    "q_yesno":   r"\b(был ли|были ли|была ли|было ли|является ли|являлся ли|можно ли|есть ли)\b",
    "q_which":   r"\b(какой|какая|какое|какие|который|которая)\b",
}
QUESTION_PATTERNS = {k: re.compile(p, re.IGNORECASE) for k, p in QUESTION_PATTERNS.items()}

def detect_question_types(q):
    if not isinstance(q, str):
        return {k: 0 for k in QUESTION_PATTERNS}
    return {k: int(bool(p.search(q))) for k, p in QUESTION_PATTERNS.items()}

RE_DIGIT       = re.compile(r"\d")
RE_YEAR        = re.compile(r"\b(1[0-9]{3}|20[0-9]{2})\b")
RE_DIGIT_RANGE = re.compile(r"\d+\s*[–\-—]\s*\d+")
RE_NUMWORD     = re.compile(
    r"\b(один|одна|одно|два|две|три|четыре|пять|шесть|семь|восемь|девять|десять|"
    r"сто|двести|триста|тысяча|миллион|миллиард)\b", re.IGNORECASE)
RE_CAP_WORD    = re.compile(r"\b([А-ЯЁ][а-яё]+)")

def text_signals(s):
    out = dict(has_digit=0, has_year=0, has_digit_range=0, has_numword=0,
               n_cap_words=0, has_cap_inside=0)
    if not isinstance(s, str) or not s:
        return out
    out["has_digit"]       = int(bool(RE_DIGIT.search(s)))
    out["has_year"]        = int(bool(RE_YEAR.search(s)))
    out["has_digit_range"] = int(bool(RE_DIGIT_RANGE.search(s)))
    out["has_numword"]     = int(bool(RE_NUMWORD.search(s)))
    cap = RE_CAP_WORD.findall(s)
    first = s.lstrip().split(maxsplit=1)
    first_word = first[0].rstrip(".,!?;:") if first else ""
    n_cap = len(cap)
    if cap and first_word and cap[0] == first_word:
        n_cap = max(0, n_cap - 1)
    out["n_cap_words"]    = n_cap
    out["has_cap_inside"] = int(n_cap > 0)
    return out

pymorphy3: on


In [10]:
tr = train.copy()
te = test.copy()
tr["_is_train"] = 1
te["_is_train"] = 0
if "relevance" not in te.columns:
    te["relevance"] = np.nan

df = pd.concat([tr, te], ignore_index=True).reset_index(drop=True)
for c in ["query_id", "document_id", "article_id"]:
    assert df[c].notna().all(), f"NaN in {c}"

df["query_clean"] = df["query"].map(clean_text)
df["title_clean"] = df["title"].map(clean_text)
df["text_clean"]  = df["text"].map(clean_text)

df["passage_index"]         = pd.to_numeric(df["passage_index"], errors="coerce")
df["article_passage_count"] = pd.to_numeric(df["article_passage_count"], errors="coerce")
df["article_char_count"]    = pd.to_numeric(df["article_char_count"], errors="coerce")
df["flag_count_negative"] = (df["article_passage_count"] < 0).astype("int8")
df.loc[df["article_passage_count"] < 0, "article_passage_count"] = np.nan
ac = df["article_char_count"]
q999 = float(np.nanpercentile(ac.dropna(), 99.9)) if ac.notna().any() else 1e6
thr  = max(q999, 1_000_000.0)
df["flag_char_count_outlier"] = (ac > thr).astype("int8")
df.loc[ac > thr, "article_char_count"] = np.nan
pi, cnt = df["passage_index"], df["article_passage_count"]
df["flag_index_out_of_range"] = (
    (pi.notna() & (pi < 0)) | (pi.notna() & cnt.notna() & (pi >= cnt))
).astype("int8")

uniq_q   = df.drop_duplicates("query_id")[["query_id", "query_clean"]]
uniq_doc = df.drop_duplicates("document_id")[["document_id", "text_clean"]]
uniq_art = df.drop_duplicates("article_id")[["article_id", "title_clean"]]

Q_TYPE_CACHE = {qid: detect_question_types(q)
                for qid, q in zip(uniq_q["query_id"], uniq_q["query_clean"])}
Q_TS_CACHE   = {did: text_signals(t)
                for did, t in zip(uniq_doc["document_id"], uniq_doc["text_clean"])}

from sentence_transformers import SentenceTransformer, CrossEncoder

def encode_unique(model, frame, id_col, text_col):
    texts = frame[text_col].fillna("").tolist()
    return dict(zip(frame[id_col].tolist(),
                    model.encode(texts, batch_size=EMB_BATCH,
                                 normalize_embeddings=True,
                                 show_progress_bar=True,
                                 convert_to_numpy=True)))

if os.path.exists(EMB_CACHE):
    with open(EMB_CACHE, "rb") as f:
        Q_EMB, DOC_EMB, ART_EMB = pickle.load(f)
    print("embeddings loaded")
else:
    print(f"loading bi-encoder: {EMB_MODEL_NAME}")
    emb_model = SentenceTransformer(EMB_MODEL_NAME, device=DEVICE)
    Q_EMB   = encode_unique(emb_model, uniq_q,   "query_id",    "query_clean")
    DOC_EMB = encode_unique(emb_model, uniq_doc, "document_id", "text_clean")
    ART_EMB = encode_unique(emb_model, uniq_art, "article_id",  "title_clean")
    with open(EMB_CACHE, "wb") as f:
        pickle.dump((Q_EMB, DOC_EMB, ART_EMB), f)
    print("embeddings saved")

if os.path.exists(CE_CACHE):
    with open(CE_CACHE, "rb") as f:
        CE_SCORES = pickle.load(f)
    print("ce scores loaded")
else:
    print(f"loading cross-encoder: {CE_MODEL_NAME}")
    ce_model = CrossEncoder(CE_MODEL_NAME, max_length=CE_MAX_LEN, device=DEVICE)
    pairs_df = df[["query_id", "document_id"]].drop_duplicates().reset_index(drop=True)
    q_map = uniq_q.set_index("query_id")["query_clean"]
    d_map = uniq_doc.set_index("document_id")["text_clean"]
    pairs = [(q_map.get(q, ""), d_map.get(d, ""))
             for q, d in zip(pairs_df["query_id"], pairs_df["document_id"])]
    ce_scores = ce_model.predict(pairs, batch_size=CE_BATCH, show_progress_bar=True)
    CE_SCORES = {(q, d): float(s) for q, d, s in
                 zip(pairs_df["query_id"], pairs_df["document_id"], ce_scores)}
    with open(CE_CACHE, "wb") as f:
        pickle.dump(CE_SCORES, f)
    print(f"ce scores saved: {len(CE_SCORES)}")

q_ser = df.drop_duplicates("query_id").set_index("query_id")["query_clean"]
query_tokens = {q: tokenize(t) for q, t in q_ser.items()}
query_char3  = {q: char3(t) for q, t in q_ser.items()}

p_ser = df.drop_duplicates("document_id").set_index("document_id")["text_clean"]
passage_tokens = {d: tokenize(t) for d, t in p_ser.items()}
passage_set    = {d: set(v) for d, v in passage_tokens.items()}
passage_tf     = {d: Counter(v) for d, v in passage_tokens.items()}
passage_char3  = {d: char3(t) for d, t in p_ser.items()}

a_ser = df.drop_duplicates("article_id").set_index("article_id")["title_clean"]
title_tokens = {a: tokenize(t) for a, t in a_ser.items()}
title_set    = {a: set(v) for a, v in title_tokens.items()}
title_tf     = {a: Counter(v) for a, v in title_tokens.items()}
title_char3  = {a: char3(t) for a, t in a_ser.items()}

idf_text,  avgdl_text  = build_idf(passage_tokens)
idf_title, avgdl_title = build_idf(title_tokens)

qid_a   = df["query_id"].to_numpy()
did_a   = df["document_id"].to_numpy()
aid_a   = df["article_id"].to_numpy()
pidx_a  = df["passage_index"].to_numpy()
pcnt_a  = df["article_passage_count"].to_numpy()
achar_a = df["article_char_count"].to_numpy()
flag_cnt_a = df["flag_count_negative"].to_numpy()
flag_ac_a  = df["flag_char_count_outlier"].to_numpy()
flag_idx_a = df["flag_index_out_of_range"].to_numpy()
src_a   = df["ingest_source"].fillna("NA").astype(str).to_numpy()
tmiss_a = (df["title"].isna() | (df["title"].astype(str).str.strip() == "")).astype(int).to_numpy()
xmiss_a = (df["text"].isna()  | (df["text"].astype(str).str.strip()  == "")).astype(int).to_numpy()
empty_tf = Counter()

QT_NAMES = list(QUESTION_PATTERNS.keys())
feat_cols = [
    "bm25_text", "bm25_title", "overlap_text", "overlap_title",
    "jac3_text", "jac3_title", "log_len_text", "log_len_title",
    "log_query_len", "query_uniq_terms",
    "passage_index", "passage_index_norm", "is_first", "is_last",
    "log_passage_count", "log_article_char", "avg_passage_char",
    "is_title_missing", "is_text_missing",
    "flag_count_negative", "flag_char_count_outlier", "flag_index_out_of_range",
    "cos_sim_text", "cos_sim_title",
    "ce_score",
    *QT_NAMES,
    "has_digit", "has_year", "has_digit_range", "has_numword",
    "n_cap_words", "has_cap_inside",
    "q_howmany_x_digit", "q_howmany_x_range", "q_howmany_x_numword",
    "q_when_x_year",
    "q_where_x_cap", "q_who_x_cap", "q_yesno_x_cap",
    "ingest_source",
]
feat = {k: [] for k in feat_cols}

for i in range(len(df)):
    qid, did, aid = qid_a[i], did_a[i], aid_a[i]
    qt = query_tokens.get(qid, [])
    qt_set = set(qt)
    dt = passage_tokens.get(did, [])
    dt_set = passage_set.get(did, set())
    tt = title_tokens.get(aid, [])
    tt_set = title_set.get(aid, set())

    feat["bm25_text"].append(bm25(qt_set, passage_tf.get(did, empty_tf), len(dt), idf_text, avgdl_text))
    feat["bm25_title"].append(bm25(qt_set, title_tf.get(aid, empty_tf), len(tt), idf_title, avgdl_title))
    feat["overlap_text"].append(len(qt_set & dt_set) / len(qt_set) if qt_set else 0.0)
    feat["overlap_title"].append(len(qt_set & tt_set) / len(qt_set) if qt_set else 0.0)
    feat["jac3_text"].append(jaccard(query_char3.get(qid, set()), passage_char3.get(did, set())))
    feat["jac3_title"].append(jaccard(query_char3.get(qid, set()), title_char3.get(aid, set())))
    feat["log_len_text"].append(math.log1p(len(dt)))
    feat["log_len_title"].append(math.log1p(len(tt)))
    feat["log_query_len"].append(math.log1p(len(qt)))
    feat["query_uniq_terms"].append(len(qt_set))

    qe, de, te = Q_EMB.get(qid), DOC_EMB.get(did), ART_EMB.get(aid)
    feat["cos_sim_text"].append(float(qe @ de) if qe is not None and de is not None else np.nan)
    feat["cos_sim_title"].append(float(qe @ te) if qe is not None and te is not None else np.nan)
    feat["ce_score"].append(CE_SCORES.get((qid, did), np.nan))

    pidx, pcnt, achar = pidx_a[i], pcnt_a[i], achar_a[i]
    feat["passage_index"].append(pidx)
    feat["passage_index_norm"].append(pidx/max(pcnt-1, 1) if (np.isfinite(pidx) and np.isfinite(pcnt)) else np.nan)
    feat["is_first"].append(int(pidx == 0) if np.isfinite(pidx) else 0)
    feat["is_last"].append(int(pidx == pcnt-1) if (np.isfinite(pidx) and np.isfinite(pcnt)) else 0)
    feat["log_passage_count"].append(safe_log1p(pcnt))
    feat["log_article_char"].append(safe_log1p(achar))
    feat["avg_passage_char"].append(achar/max(pcnt, 1) if (np.isfinite(achar) and np.isfinite(pcnt)) else np.nan)
    feat["is_title_missing"].append(tmiss_a[i])
    feat["is_text_missing"].append(xmiss_a[i])
    feat["flag_count_negative"].append(int(flag_cnt_a[i]))
    feat["flag_char_count_outlier"].append(int(flag_ac_a[i]))
    feat["flag_index_out_of_range"].append(int(flag_idx_a[i]))

    qtypes = Q_TYPE_CACHE.get(qid, {k: 0 for k in QT_NAMES})
    ts     = Q_TS_CACHE.get(did, dict(has_digit=0, has_year=0, has_digit_range=0,
                                      has_numword=0, n_cap_words=0, has_cap_inside=0))
    for name in QT_NAMES:
        feat[name].append(qtypes.get(name, 0))
    feat["has_digit"].append(ts["has_digit"])
    feat["has_year"].append(ts["has_year"])
    feat["has_digit_range"].append(ts["has_digit_range"])
    feat["has_numword"].append(ts["has_numword"])
    feat["n_cap_words"].append(ts["n_cap_words"])
    feat["has_cap_inside"].append(ts["has_cap_inside"])

    # crosses
    feat["q_howmany_x_digit"].append(qtypes.get("q_howmany", 0) * ts["has_digit"])
    feat["q_howmany_x_range"].append(qtypes.get("q_howmany", 0) * ts["has_digit_range"])
    feat["q_howmany_x_numword"].append(qtypes.get("q_howmany", 0) * ts["has_numword"])
    feat["q_when_x_year"].append(qtypes.get("q_when", 0) * ts["has_year"])
    feat["q_where_x_cap"].append(qtypes.get("q_where", 0) * ts["has_cap_inside"])
    feat["q_who_x_cap"].append(qtypes.get("q_who", 0) * ts["has_cap_inside"])
    feat["q_yesno_x_cap"].append(qtypes.get("q_yesno", 0) * ts["has_cap_inside"])

    feat["ingest_source"].append(src_a[i])

loading bi-encoder: sergeyzh/rubert-tiny-turbo


modules.json:   0%|          | 0.00/368 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/431k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/712 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  117MB            

... (19 outputs left out)

tokenizer_config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/4.75M [00:00<?, ?B/s]

Batches:   0%|          | 0/704 [00:00<?, ?it/s]

ce scores saved: 45037


In [11]:
X_all = pd.DataFrame(feat)
X_all = pd.get_dummies(X_all, columns=["ingest_source"], prefix="src", dummy_na=False)
bool_cols = [c for c in X_all.columns if X_all[c].dtype == bool]
if bool_cols:
    X_all[bool_cols] = X_all[bool_cols].astype("int8")

train_mask = df["_is_train"].to_numpy() == 1
Xtr = X_all.loc[train_mask].reset_index(drop=True)
Xte = X_all.loc[~train_mask].reset_index(drop=True)
y   = df.loc[train_mask, "relevance"].astype(int).to_numpy()
groups   = df.loc[train_mask, "query_id"].to_numpy()
test_ids = df.loc[~train_mask, "id"].to_numpy()

print(f"train={len(Xtr)} test={len(Xte)} queries={len(np.unique(groups))} "
      f"pos/query={y.sum()/len(np.unique(groups)):.2f} features={Xtr.shape[1]}")

train=31530 test=13507 queries=3042 pos/query=2.69 features=49


In [12]:
base_params = dict(
    objective="lambdarank", metric="ndcg",
    boosting_type="gbdt", n_estimators=2000,
    learning_rate=0.05, num_leaves=31, min_child_samples=20,
    subsample=0.8, subsample_freq=1, colsample_bytree=0.8,
    reg_lambda=1.0, random_state=SEED, n_jobs=-1, verbose=-1,
)
gkf = GroupKFold(n_splits=N_FOLDS)
fold_scores, test_pred_sum, models = [], np.zeros(len(Xte)), []

for fold, (tr_idx, va_idx) in enumerate(gkf.split(Xtr, y, groups=groups)):
    X_tr, y_tr, g_tr = Xtr.iloc[tr_idx], y[tr_idx], groups[tr_idx]
    X_va, y_va, g_va = Xtr.iloc[va_idx], y[va_idx], groups[va_idx]
    o  = np.argsort(g_tr, kind="stable"); X_tr, y_tr, g_tr = X_tr.iloc[o], y_tr[o], g_tr[o]
    _, cnt_tr = np.unique(g_tr, return_counts=True)
    ov = np.argsort(g_va, kind="stable"); X_va, y_va, g_va = X_va.iloc[ov], y_va[ov], g_va[ov]
    _, cnt_va = np.unique(g_va, return_counts=True)
    m = lgb.LGBMRanker(eval_at=[K], **base_params)
    m.fit(X_tr, y_tr, group=cnt_tr,
          eval_set=[(X_va, y_va)], eval_group=[cnt_va],
          callbacks=[lgb.early_stopping(100, verbose=False)])
    models.append(m)
    pv = m.predict(X_va)
    val_df = pd.DataFrame({"q": g_va, "y": y_va, "p": pv})
    ndcgs = [ndcg_at_k(g["y"].to_numpy(), g["p"].to_numpy(), K)
             for _, g in val_df.groupby("q", sort=False)]
    ndcgs = [v for v in ndcgs if v is not None]
    fs = float(np.mean(ndcgs)); fold_scores.append(fs)
    print(f"fold {fold}: NDCG@10={fs:.4f} (q={len(ndcgs)})")
    test_pred_sum += m.predict(Xte)

print(f"CV NDCG@10 = {np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}")

imp = pd.Series(models[0].feature_importances_, index=Xtr.columns).sort_values(ascending=False)
print("\ntop-20 features:\n" + imp.head(20).to_string())

/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


fold 0: NDCG@10=0.8380 (q=608)


/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


fold 1: NDCG@10=0.8383 (q=609)


/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


fold 2: NDCG@10=0.8309 (q=609)


... (2 outputs left out)

/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")
/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


fold 4: NDCG@10=0.8484 (q=608)


/usr/local/lib/python3.13/dist-packages/lightgbm/sklearn.py:861: UserWarning: Found 'eval_at' in params. Will use it instead of 'eval_at' argument
  _log_warning(f"Found '{alias}' in params. Will use it instead of 'eval_at' argument")


CV NDCG@10 = 0.8395 ± 0.0057

top-20 features:
bm25_text             425
ce_score              388
log_len_text          367
passage_index_norm    339
cos_sim_text          328
log_article_char      323
cos_sim_title         267
log_passage_count     241
passage_index         238
avg_passage_char      228
jac3_text             215
bm25_title            197
overlap_text          190
n_cap_words           186
jac3_title            147
log_len_title         124
overlap_title          45
log_query_len          43
query_uniq_terms       28
has_year               22


In [13]:
test_pred = test_pred_sum / len(models)
sub = pd.DataFrame({"id": test_ids, "prediction": test_pred})
sub["prediction"] = sub["prediction"].fillna(0.0)
sub.to_csv(OUT_PATH, index=False)
print(f"saved {OUT_PATH}: {sub.shape}")

saved submission.csv: (13507, 2)
